In [ ]:
import pandas as pd
import re

# Data Loading

In this step, the dataset is loaded from a CSV file into a pandas DataFrame. This dataset contains job postings labeled as real or fraudulent.

In [ ]:
# Load your dataset
df_raw = pd.read_csv('fake_job_postings.csv')

# Preview data
df_raw.head()

# Initial Data Exploration

We perform an initial inspection of the dataset to understand its structure, data types, and identify missing values.

Key checks:
- Data shape
- Column types
- Missing values distribution
- Target variable balance (fraudulent vs non-fraudulent)

In [ ]:
df_raw.shape

In [ ]:
# Check missing values
missing_values = df_raw.isnull().sum()

print("Missing values per column:")
print(missing_values)

In [ ]:
df_raw['location'].value_counts().head(20)

In [ ]:
df_raw['fraudulent'].value_counts().head(20)

In [ ]:
df_raw['country'] = df_raw['location'].str.split(',').str[0].str.strip()

df_raw['country'].value_counts().head(20)

In [ ]:
fraud_df = df_raw[df_raw['fraudulent'] == 1]

fraud_by_country = fraud_df['country'].value_counts()

fraud_by_country.head(10)

# Data Cleaning & Preparation

A copy of the raw dataset is created to preserve the original data. Irrelevant columns such as 'salary_range' and 'department' are removed as they are not essential for text-based classification.

In [ ]:
# Drop rows with missing values
df_clean_desc = df_raw.dropna(subset=['description'])


print("Shape before cleaning:", df_raw.shape)
print("Shape after cleaning:", df_clean_desc.shape)
print(df_clean_desc.columns)

In [ ]:
drop_cols = ['salary_range', 'department']
df_clean = df_clean_desc.drop(columns=drop_cols)

print("Before:", df_clean_desc.shape)
print("After:", df_clean.shape)

# Filtering US Job Postings

The dataset is filtered to include only job postings from the United States. This ensures consistency in language and reduces noise from international variations.

In [ ]:
# Filter dataset to include only US job postings
df_us = df_clean[df_clean['country'] == 'US'].copy()

print(df_us.shape)
print(df_us.columns)

# Handling Missing Values

Missing values in textual columns are filled with the string "missing". This ensures that no data is lost while preserving information about missing fields, which may be indicative of fraudulent postings.

In [ ]:
# Check missing values
missing_values_us = df_us.isnull().sum()

print("Missing values per column:")
print(missing_values_us)

In [ ]:
df_us['fraudulent'].value_counts().head(20)

In [ ]:
# Replace missing values with 'missing' to retain information
text_cols = ['title', 'description', 'company_profile', 'requirements', 'benefits']

df_us[text_cols] = df_us[text_cols].fillna('missing')

cat_cols = ['employment_type','required_experience', 'required_education', 'industry', 'function']

df_us[cat_cols] = df_us[cat_cols].fillna('unknown')

In [ ]:
# Check missing values
missing_values_us = df_us.isnull().sum()

print("Missing values per column:")
print(missing_values_us)

# Feature Engineering

Additional features are created to enhance the model's ability to detect fraudulent job postings:

- Missing indicators: Flags for missing 'requirements', 'company_profile', and 'benefits'
- has_number: Detects presence of numbers in job descriptions (potential scam indicator)
- desc_length: Measures the length of job descriptions (fake jobs tend to be shorter)

In [ ]:
df_us['missing_requirements'] = (df_us['requirements'] == "missing").astype(int)
df_us['missing_company_profile'] = (df_us['company_profile'] == "missing").astype(int)
df_us['missing_benefits'] = (df_us['benefits'] == "missing").astype(int)

df_us['has_number'] = df_us['description'].str.contains(r'\d').astype(int)

# Feature: length of job description (fake jobs tend to be shorter)
df_us['desc_length'] = df_us['description'].str.split().str.len()

df_us[['missing_requirements','missing_company_profile','missing_benefits']].mean()

# Text Preprocessing

Text cleaning is applied to normalize the data:
- Convert text to lowercase
- Remove newline characters
- Remove extra whitespace
- Remove URLs
- Remove HTML

Minimal preprocessing is used to preserve semantic meaning, which is important for transformer-based models like BERT.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    return text.strip()

us_cols = [
    'title',
    'company_profile',
    'description',
    'requirements',
    'benefits',
    'industry',
    'function'
]

for col in us_cols:
    df_us[col] = df_us[col].apply(clean_text)

In [ ]:
# Combine all relevant text fields into a single input for BERT
df_us['full_text'] = (
    "TITLE: " + df_us['title'] + " " +
    "DESC: " + df_us['description'] + " " +
    "REQ: " + df_us['requirements'] + " " +
    "COMPANY: " + df_us['company_profile'] + " " +
    "BENEFITS: " + df_us['benefits'] + " " +
    "INDUSTRY: " + df_us['industry'] + " " +
    "FUNCTION: " + df_us['function']
)

# Save Processed Dataset

The cleaned and processed dataset is saved for use in model training and evaluation.

In [ ]:
# Save cleaned dataset
df_us.to_csv('job_scam_cleaned.csv', index=False)

# Tokenization

Tokenization is performed using the pre-trained tokenizer associated with the transformer model, ensuring compatibility with BERT and RoBERTa input formats

BERT Tokenization

In [ ]:
from transformers import BertTokenizer

tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')

encoded_bert = tokenizer_bert(
    df_us['full_text'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

BERT Validation

In [ ]:
print(encoded_bert[0])

In [ ]:
print(type(encoded_bert))

In [ ]:
print("Keys:", encoded_bert.keys())

print("Input IDs shape:", encoded_bert['input_ids'].shape)
print("Attention mask shape:", encoded_bert['attention_mask'].shape)

labels = df_us['fraudulent'].values
print("Labels length:", len(labels))

assert encoded_bert['input_ids'].shape[0] == len(labels)

print("Sample input length:", len(encoded_bert['input_ids'][0]))
print("BERT Tokenization validation passed!")

RoBERTa Tokenization

In [ ]:
from transformers import RobertaTokenizer

tokenizer_roberta = RobertaTokenizer.from_pretrained('roberta-base')

encoded_roberta = tokenizer_roberta(
    df_us['full_text'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

RoBERTa Validation

In [ ]:
print(encoded_roberta[0])

In [ ]:
print(type(encoded_roberta))

In [ ]:
print("Keys:", encoded_roberta.keys())

print("Input IDs shape:", encoded_roberta['input_ids'].shape)
print("Attention mask shape:", encoded_roberta['attention_mask'].shape)

labels = df_us['fraudulent'].values
print("Labels length:", len(labels))

assert encoded_roberta['input_ids'].shape[0] == len(labels)

print("Sample input length:", len(encoded_roberta['input_ids'][0]))
print("RoBERTa Tokenization validation passed!")